# Notebook 08 — Energy encoder replication with 10 fresh paired seeds

**Run All launches the actual detached experiment.** It returns while the worker runs independently in tmux. Expected runtime is 7–8 hours, with an enforced 8-hour cap from first launch, including calibration, resume checks, training and evaluation. No result is claimed before completion.

## Registered question and protocol

Does the unchanged FMM-inspired energy encoder differ from ViT in final Food101 development accuracy after 350 updates? The primary endpoint is the mean paired difference (energy minus ViT) across 10 fresh training seeds. Use a two-sided paired t-test at α = .05 and a 95% interval with 9 degrees of freedom. A positive significant difference supports higher accuracy; a negative significant difference supports lower accuracy. Lack of significance remains unresolved.

- **20 training runs:** each model uses seeds 20261101–20261110, 350 successful updates, 44,800 image exposures, physical/effective batch 128, 128px images, patch 8, width 192 and FP32. Seed pairs share the exact patch stem, final norm and predictor initialization and image/augmentation/mask streams. Data and mask stream seeds are also new. Architectures and parameter counts differ, as in the pilot.
- **Unchanged energy model:** same tagged Notebook 07 encoder; 4 refinement steps, η = .5, λnear = λfar = .25, summary width 32. Both online and EMA encoder branches use it; the existing predictor remains.
- **40 evaluations:** initialized and trained EMA encoders each receive the same fixed 20-epoch linear probe. Own-learning, adjusted gains, compute/memory and representation diagnostics are secondary exploratory results. All 20 arms and 40 probes are required; no favorable subset or stopping at significance.
- **Pilot excluded:** the old two seed pairs are planning evidence only. Power planning targeted a true 0.5 percentage-point difference with 90% power, assuming the pilot SD transfers; this is uncertain, not a guarantee. The development set has been reused; this confirms across training seeds, not a new test population.

## Inputs, outputs and recovery

Inputs: existing sealed Food101 set of 10,100 pretraining/probe-training images and 2,020 development images. Official test data stays reserved. An explicitly separate evaluation smoke may use audited old feature caches as validation-only fixtures; they never enter new study outcomes.

Outputs: `artifacts/notebook_08/energy-replication-001/`. Final report: `stages/learning/stage_results.md`. Full resumable checkpoints: `checkpoints/energy-replication-001/`, with retention of the latest two and best checkpoints, saves at epoch/stage boundaries and at least every 50 updates or 15 minutes, atomic manifests, every RNG/data position, optimizer/scheduler/scaler and one writer.

Exact resume: use the `ahr-go` kernel and Run All unchanged with the same RUN_ID before the original deadline. It reconnects to a live worker or resumes verified checkpoints. Invalid state fails explicitly; the clock never resets. A cutoff leaves INCOMPLETE and retains valid checkpoints. Workspace storage is persistent `/dev/sda1`; instance-deletion policy remains unverified.

Attach with `tmux attach -t fmm-replication-energy-replication-001`. Detach with Ctrl-b, then d. The independent deadline watcher terminates only this study’s named process tree.


## 1. Load tagged definitions

In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
from pathlib import Path
import nbformat, importlib.util, sys
ROOT = Path("/home/benjamin/workspace/ahr_go/fmm_jeppa").resolve()
_boot = {}
for _cell in nbformat.read(ROOT / "fmm_jepa/bootstrap.ipynb", as_version=4).cells:
    if _cell.cell_type == "code" and "ahr-export" in _cell.metadata.get("tags", []):
        exec(compile(_cell.source, "bootstrap#ahr-export", "exec"), _boot)
_boot["install_notebook_package"](ROOT)
_spec = importlib.util.spec_from_loader("fmm08_workflow", _boot["TaggedNotebookLoader"](ROOT / "experiments/08_replication/workflow.ipynb"))
WORKFLOW = importlib.util.module_from_spec(_spec)
sys.modules[_spec.name] = WORKFLOW
_spec.loader.exec_module(WORKFLOW)


## 2. Fixed registered configuration

In [ ]:
RUN_ID = "energy-replication-001"
CONFIG = WORKFLOW.study_configuration(RUN_ID)
CONFIG

## 3. Launch or reconnect
The worker verifies actual timing, full-state 4-step versus 2+2-step resume, and the evaluation integration before beginning the long comparison.

In [ ]:
LAUNCH = WORKFLOW.launch_study(ROOT, CONFIG)
LAUNCH

## 4. Inspect progress
Rerun this cell for the current stage, checkpoint counts, and original hard-stop time.

In [ ]:
STATUS = WORKFLOW.inspect_study(ROOT, RUN_ID)
STATUS

## Interpretation
The primary accuracy comparison concerns this fixed 350-update regime. More seeds estimate variability; they do not make the algorithm better. A longer-training claim would require a different predeclared study. Mathematical stability and nonconstant features are not evidence of superior learned representations. Compute results describe the current implementation/hardware, not an asymptotic scaling proof.